In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.metrics import roc_auc_score

PROJECT_ROOT = Path("/mnt/c/dev/my_ml_project")

DATA_DIR = PROJECT_ROOT / "data"
OOF_DIR = PROJECT_ROOT / "oof_preds"
SUB_DIR = PROJECT_ROOT / "submissions"

TRAIN_PATH = DATA_DIR / "train.csv"
TARGET = "임신 성공 여부"

SAVE_DIR = PROJECT_ROOT / "analysis" / "oof_slice"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TRAIN_PATH exists:", TRAIN_PATH.exists())
print("SAVE_DIR:", SAVE_DIR)

PROJECT_ROOT: /mnt/c/dev/my_ml_project
TRAIN_PATH exists: True
SAVE_DIR: /mnt/c/dev/my_ml_project/analysis/oof_slice


In [2]:
def data_preprocessing(df):
    df = df.copy()
    time_cols = [
        '임신 시도 또는 마지막 임신 경과 연수',
        '난자 해동 경과일',
        '난자 혼합 경과일',
        '배아 이식 경과일',
        '배아 해동 경과일'
    ]

    for col in time_cols:
        df[f'{col}_performed'] = (
            df[col].notnull()
        ).astype(int)


    df = df.fillna(0)
    infertility_cols = [
        '불임 원인 - 난관 질환',
        '불임 원인 - 남성 요인',
        '불임 원인 - 배란 장애',
        '불임 원인 - 여성 요인',
        '불임 원인 - 자궁경부 문제',
        '불임 원인 - 자궁내막증',
        '불임 원인 - 정자 농도',
        '불임 원인 - 정자 운동성',
        '불임 원인 - 정자 형태',
        '불임 원인 - 정자 면역학적 요인'
    ]

    male_cols = [
        '불임 원인 - 남성 요인',
        '불임 원인 - 정자 농도',
        '불임 원인 - 정자 운동성',
        '불임 원인 - 정자 형태',
        '불임 원인 - 정자 면역학적 요인'
    ]

    female_cols = [
        '불임 원인 - 난관 질환',
        '불임 원인 - 배란 장애',
        '불임 원인 - 여성 요인',
        '불임 원인 - 자궁경부 문제',
        '불임 원인 - 자궁내막증'
    ]

    df['불임원인_총개수'] = df[infertility_cols].sum(axis=1)

    df['남성_원인_수'] = df[male_cols].sum(axis=1)

    df['여성_원인_수'] = df[female_cols].sum(axis=1)

    df['남녀_복합_원인'] = (
        (df['남성_원인_수'] > 0) &
        (df['여성_원인_수'] > 0)
    ).astype(int)

    df['원인불명'] = (
        df['불임원인_총개수'] == 0
    ).astype(int)

    count_cols = [
        '총 시술 횟수',
        'IVF 시술 횟수',
        'DI 시술 횟수',
        '총 임신 횟수',
        'IVF 임신 횟수',
        'DI 임신 횟수',
        '총 출산 횟수',
        'IVF 출산 횟수',
        'DI 출산 횟수',
        '클리닉 내 총 시술 횟수'
    ]

    count_map = {
        '0회': 0,
        '1회': 1,
        '2회': 2,
        '3회': 3,
        '4회': 4,
        '5회': 5,
        '6회 이상': 6,
    }

    for col in count_cols:
        df[col] = df[col].map(count_map).astype(float)

    df['고령여부'] = df['시술 당시 나이'].isin([
        '만38-39세',
        '만40-42세',
        '만43-44세',
        '만45-50세'
    ]).astype(int)


    df['배아_생성률'] = np.where(
        df['혼합된 난자 수'] == 0,
        0,
        df['총 생성 배아 수'] / df['혼합된 난자 수']
    )

    # 2. 배아 이식 효율
    df['배아_이식률'] = np.where(
        df['총 생성 배아 수'] == 0,
        0,
        df['이식된 배아 수'] / df['총 생성 배아 수']
    )

    # 3. 배아 냉동 비율
    df['배아_냉동률'] = np.where(
        df['총 생성 배아 수'] == 0,
        0,
        df['저장된 배아 수'] / df['총 생성 배아 수']
    )
    df['IVF_임신성공률'] = np.where(
        df['IVF 시술 횟수'] == 0,
        0,
        df['IVF 임신 횟수'] / df['IVF 시술 횟수']
    )

    df['DI_임신성공률'] = np.where(
        df['DI 시술 횟수'] == 0,
        0,
        df['DI 임신 횟수'] / df['DI 시술 횟수']
    )

    df['고령_난자수_interaction'] = (
        df['고령여부'] *
        df['수집된 신선 난자 수']
    )

    df['배아이식_수행여부'] = (
        df['이식된 배아 수'] > 0
    ).astype(int)

    df['배아_이식_집중도'] = np.where(
        (df['이식된 배아 수'] + df['저장된 배아 수']) == 0,
        0,
        df['이식된 배아 수'] /
        (
            df['이식된 배아 수'] +
            df['저장된 배아 수']
        )
    )
    df["배아 생성 주요 이유"] = (
        df["배아 생성 주요 이유"]
        .astype(str)
        .astype("category")
    )
    df["특정 시술 유형"] = (
        df["특정 시술 유형"]
        .astype(str)
        .astype("category")
    )

    # 고령 × 이식 배아 수
    df['고령_배아이식'] = (
        df['고령여부'] *
        df['이식된 배아 수']
    )

    # 고령 × 총 생성 배아 수
    df['고령_배아생성'] = (
        df['고령여부'] *
        df['총 생성 배아 수']
    )

    # 고령 × 저장 배아 수
    df['고령_배아저장'] = (
        df['고령여부'] *
        df['저장된 배아 수']
    )

    # 고령 × 미세주입 난자 수
    df['고령_미세주입난자'] = (
        df['고령여부'] *
        df['미세주입된 난자 수']
    )

    df['출산_임신_전환율'] = np.where(
        df['총 임신 횟수'] == 0,
        0,
        df['총 출산 횟수'] / df['총 임신 횟수']
    )

    df['클리닉_집중도'] = np.where(
        df['총 시술 횟수'] == 0,
        0,
        df['클리닉 내 총 시술 횟수'] / df['총 시술 횟수']
    )

    df['첫_시술_여부'] = (
        df['총 시술 횟수'] == 0
    ).astype(int)



    binary_keywords = [
        "코드", "나이", "유형", "여부", "원인", "이유", "횟수", "출처"
    ]

    binary_cols = [
        col for col in df.columns
        if any(keyword in col for keyword in binary_keywords)
    ]

    df[binary_cols] = df[binary_cols].astype('category')

    object_cols = df.select_dtypes(include="object").columns

    df[object_cols] = df[object_cols].astype(str)

    cat_cols = df.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    for col in cat_cols:
        if col != TARGET:
            df[col] = df[col].astype(str)

    drop_cols = [
        '배아이식_수행여부'
    ]
    df = df.drop(
        columns=drop_cols
        )
    return df

In [3]:
train_raw = pd.read_csv(TRAIN_PATH)

y = train_raw[TARGET].astype(int).reset_index(drop=True)
X_raw = train_raw.drop(columns=[TARGET]).reset_index(drop=True)

print("X_raw:", X_raw.shape)
print("y:", y.shape)
print("target mean:", y.mean())

X_raw: (256351, 68)
y: (256351,)
target mean: 0.2583489044318142


In [4]:
seed42_oof = np.load(OOF_DIR / "combo_te_s10" / "final_combo_rank.npy")
seed2024_oof = np.load(OOF_DIR / "combo_te_v1_s10_seed2024" / "final_oof_seed2024.npy")
seed777_oof = np.load(OOF_DIR / "combo_te_v1_s10_seed777" / "final_oof_seed777.npy")
seed999_oof = np.load(OOF_DIR / "combo_te_v1_s10_seed999" / "final_oof_seed999.npy")

champion_oof = (
    seed42_oof +
    seed2024_oof +
    seed777_oof +
    seed999_oof
) / 4

print("seed42:", roc_auc_score(y, seed42_oof))
print("seed2024:", roc_auc_score(y, seed2024_oof))
print("seed777:", roc_auc_score(y, seed777_oof))
print("seed999:", roc_auc_score(y, seed999_oof))
print("champion 4-seed:", roc_auc_score(y, champion_oof))

seed42: 0.7404963277840849
seed2024: 0.7405675120287843
seed777: 0.7404199005438702
seed999: 0.7404124217483375
champion 4-seed: 0.7407705074934163


In [5]:
analysis_df = X_raw.copy()
analysis_df["target"] = y
analysis_df["pred"] = champion_oof

analysis_df["abs_error"] = np.abs(analysis_df["target"] - analysis_df["pred"])
analysis_df["pred_rank"] = pd.Series(champion_oof).rank(pct=True).values

analysis_df.head()

,ID,시술 시기 코드,시술 당시 나이,임신 시도 또는 마지막 임신 경과 연수,시술 유형,특정 시술 유형,배란 자극 여부,배란 유도 유형,단일 배아 이식 여부,착상 전 유전 검사 사용 여부,...,PGS 시술 여부,난자 채취 경과일,난자 해동 경과일,난자 혼합 경과일,배아 이식 경과일,배아 해동 경과일,target,pred,abs_error,pred_rank
0,TRAIN_000000,TRZKPL,만18-34세,NaN,IVF,ICSI,1,기록되지 않은 시행,0.0,NaN,...,NaN,0.0,NaN,0.0,3.0,NaN,0,0.918868,0.918868,0.920788
1,TRAIN_000001,TRYBLT,만45-50세,NaN,IVF,ICSI,0,알 수 없음,0.0,NaN,...,NaN,0.0,NaN,0.0,NaN,NaN,0,0.025578,0.025578,0.017102
2,TRAIN_000002,TRVNRY,만18-34세,NaN,IVF,IVF,1,기록되지 않은 시행,0.0,NaN,...,NaN,0.0,NaN,0.0,2.0,NaN,0,0.673969,0.673969,0.674708
3,TRAIN_000003,TRJXFG,만35-37세,NaN,IVF,ICSI,1,기록되지 않은 시행,0.0,NaN,...,NaN,0.0,NaN,0.0,NaN,NaN,0,0.095958,0.095958,0.101556
4,TRAIN_000004,TRVNRY,만18-34세,NaN,IVF,ICSI,1,기록되지 않은 시행,0.0,NaN,...,NaN,0.0,NaN,0.0,3.0,NaN,0,0.621466,0.621466,0.621234


In [6]:
X_proc = data_preprocessing(X_raw).reset_index(drop=True)

# 보고 싶은 engineered feature만 analysis_df에 붙이기
engineered_slice_cols = [
    "배아_이식_집중도",
    "배아_생성률",
    "배아_이식률",
    "배아_냉동률",
    "IVF_임신성공률",
    "DI_임신성공률",
    "고령여부",
    "고령_난자수_interaction",
    "출산_임신_전환율",
    "클리닉_집중도",
    "배아 이식 경과일_performed",
    "배아 해동 경과일_performed",
    "난자 혼합 경과일_performed",
]

for col in engineered_slice_cols:
    if col in X_proc.columns:
        analysis_df[col] = X_proc[col].values

print("analysis_df:", analysis_df.shape)

/tmp/ipykernel_757/835862898.py:209: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df.select_dtypes(include="object").columns


analysis_df: (256351, 85)


In [7]:
def slice_auc_report(
    df,
    col,
    target_col="target",
    pred_col="pred",
    min_count=500,
    top_n=30,
):
    rows = []

    temp = df[[col, target_col, pred_col]].copy()
    temp[col] = temp[col].astype(str).fillna("NaN")

    for value, g in temp.groupby(col):
        n = len(g)
        pos = int(g[target_col].sum())
        neg = n - pos

        if n < min_count:
            continue

        # AUC는 target이 양/음 둘 다 있어야 계산 가능
        if g[target_col].nunique() < 2:
            auc = np.nan
        else:
            auc = roc_auc_score(g[target_col], g[pred_col])

        rows.append({
            "col": col,
            "value": value,
            "count": n,
            "positive": pos,
            "negative": neg,
            "positive_rate": g[target_col].mean(),
            "pred_mean": g[pred_col].mean(),
            "calibration_gap": g[pred_col].mean() - g[target_col].mean(),
            "auc": auc,
            "pred_std": g[pred_col].std(),
        })

    result = pd.DataFrame(rows)

    if result.empty:
        return result

    result = result.sort_values(
        ["auc", "count"],
        ascending=[True, False],
        na_position="last"
    )

    return result.head(top_n)



In [10]:
categorical_slice_cols = [
    "시술 시기 코드",
    "시술 유형",
    "특정 시술 유형",
    "배란 유도 유형",
    "배란 자극 여부",
    "시술 당시 나이",
    "난자 출처",
    "정자 출처",
    "난자 기증자 나이",
    "정자 기증자 나이",
    "배아 생성 주요 이유",
    "신선 배아 사용 여부",
    "동결 배아 사용 여부",
    "기증 배아 사용 여부",
    "대리모 여부",
]

categorical_slice_cols = [
    col for col in categorical_slice_cols
    if col in analysis_df.columns
]

all_cat_reports = []

for col in categorical_slice_cols:
    print("\n" + "=" * 80)
    print("Slice:", col)

    report = slice_auc_report(
        analysis_df,
        col=col,
        min_count=500,
        top_n=30,
    )

    display(report)

    if not report.empty:
        all_cat_reports.append(report)

cat_report_df = pd.concat(all_cat_reports, ignore_index=True)
cat_report_df.to_csv(SAVE_DIR / "categorical_slice_auc_report.csv", index=False)

display(
    cat_report_df.sort_values(
        ["auc", "count"],
        ascending=[True, False],
        na_position="last"
    ).head(50)
)

def numeric_bin_report(
    df,
    col,
    bins=5,
    target_col="target",
    pred_col="pred",
    min_count=500,
):
    temp = df[[col, target_col, pred_col]].copy()
    numeric = pd.to_numeric(temp[col], errors="coerce")

    bin_col = f"{col}_bin"

    # unique가 적으면 그대로 category 취급하되,
    # Int64로 강제 변환하지 않는다.
    if numeric.nunique(dropna=True) <= 10:
        temp[bin_col] = numeric.round(6).astype(str)
    else:
        try:
            temp[bin_col] = pd.qcut(
                numeric,
                q=bins,
                duplicates="drop"
            ).astype(str)
        except Exception:
            temp[bin_col] = pd.cut(
                numeric,
                bins=bins
            ).astype(str)

    temp[bin_col] = temp[bin_col].replace("nan", "NaN").fillna("NaN")

    return slice_auc_report(
        temp,
        col=bin_col,
        target_col=target_col,
        pred_col=pred_col,
        min_count=min_count,
        top_n=50,
    )


Slice: 시술 시기 코드


,col,value,count,positive,negative,positive_rate,pred_mean,calibration_gap,auc,pred_std
6,시술 시기 코드,TRZKPL,35544,9068,26476,0.255120,0.494128,0.239008,0.724789,0.266968
2,시술 시기 코드,TRJXFG,36031,9594,26437,0.266271,0.514627,0.248356,0.727333,0.282896
4,시술 시기 코드,TRXQMD,34831,8921,25910,0.256122,0.490376,0.234254,0.727659,0.265446
3,시술 시기 코드,TRVNRY,36173,9397,26776,0.259779,0.503665,0.243885,0.728418,0.275126
5,시술 시기 코드,TRYBLT,36713,9879,26834,0.269087,0.519180,0.250093,0.740738,0.294623
0,시술 시기 코드,TRCMWS,38090,9805,28285,0.257417,0.497704,0.240287,0.759202,0.307546
1,시술 시기 코드,TRDQAZ,38969,9564,29405,0.245426,0.481219,0.235793,0.767976,0.310950



Slice: 시술 유형


,col,value,count,positive,negative,positive_rate,pred_mean,calibration_gap,auc,pred_std
0,시술 유형,DI,6291,811,5480,0.128914,0.249524,0.120610,0.685870,0.101038
1,시술 유형,IVF,250060,65417,184643,0.261605,0.506303,0.244698,0.738864,0.287975



Slice: 특정 시술 유형


...


Slice: 기증 배아 사용 여부


,col,value,count,positive,negative,positive_rate,pred_mean,calibration_gap,auc,pred_std
1,기증 배아 사용 여부,1.0,2458,805,1653,0.327502,0.632679,0.305177,0.683346,0.231651
2,기증 배아 사용 여부,NaN,6291,811,5480,0.128914,0.249524,0.120610,0.685870,0.101038
0,기증 배아 사용 여부,0.0,247602,64612,182990,0.260951,0.505049,0.244098,0.739344,0.288202



Slice: 대리모 여부


,col,value,count,positive,negative,positive_rate,pred_mean,calibration_gap,auc,pred_std
1,대리모 여부,1.0,1049,303,746,0.288847,0.568480,0.279634,0.616803,0.198489
2,대리모 여부,NaN,6291,811,5480,0.128914,0.249524,0.120610,0.685870,0.101038
0,대리모 여부,0.0,249011,65114,183897,0.261490,0.506042,0.244551,0.739285,0.288265


,col,value,count,positive,negative,positive_rate,pred_mean,calibration_gap,auc,pred_std
45,배아 생성 주요 이유,배아 저장용,9192,8,9184,0.000870,0.042063,0.041192,0.448688,0.026437
60,대리모 여부,1.0,1049,303,746,0.288847,0.568480,0.279634,0.616803,0.198489
9,특정 시술 유형,ICSI / BLASTOCYST,1609,574,1035,0.356743,0.686063,0.329320,0.644350,0.235520
51,신선 배아 사용 여부,0.0,39924,9166,30758,0.229586,0.429630,0.200043,0.653818,0.166171
54,동결 배아 사용 여부,1.0,40126,9205,30921,0.229402,0.429068,0.199666,0.654270,0.166247
10,특정 시술 유형,Unknown,26939,6390,20549,0.237203,0.444176,0.206973,0.656616,0.172311
11,특정 시술 유형,ICSI / AH,769,167,602,0.217165,0.408653,0.191488,0.670838,0.190198
34,난자 기증자 나이,만31-35세,6366,1943,4423,0.305215,0.596870,0.291655,0.670852,0.246755
46,배아 생성 주요 이유,"기증용, 현재 시술용",3784,1437,2347,0.379757,0.709398,0.329642,0.671984,0.268099
35,난자 기증자 나이,만26-30세,4976,1733,3243,0.348272,0.640814,0.292543,0.678621,0.255231


In [11]:
numeric_slice_cols = [
    "이식된 배아 수",
    "총 생성 배아 수",
    "저장된 배아 수",
    "해동된 배아 수",
    "수집된 신선 난자 수",
    "혼합된 난자 수",
    "미세주입된 난자 수",
    "파트너 정자와 혼합된 난자 수",
    "배아 이식 경과일",
    "배아 해동 경과일",
    "난자 혼합 경과일",
    "난자 해동 경과일",
    "총 시술 횟수",
    "IVF 시술 횟수",
    "DI 시술 횟수",
    "총 임신 횟수",
    "총 출산 횟수",
]

numeric_slice_cols += [
    col for col in engineered_slice_cols
    if col in analysis_df.columns
]

numeric_slice_cols = [
    col for col in numeric_slice_cols
    if col in analysis_df.columns
]

all_num_reports = []

for col in numeric_slice_cols:
    print("\n" + "=" * 80)
    print("Numeric Slice:", col)

    report = numeric_bin_report(
        analysis_df,
        col=col,
        bins=5,
        min_count=500,
    )

    display(report)

    if not report.empty:
        report["source_numeric_col"] = col
        all_num_reports.append(report)

num_report_df = pd.concat(all_num_reports, ignore_index=True)
num_report_df.to_csv(SAVE_DIR / "numeric_slice_auc_report.csv", index=False)

display(
    num_report_df.sort_values(
        ["auc", "count"],
        ascending=[True, False],
        na_position="last"
    ).head(50)
)


Numeric Slice: 이식된 배아 수


,col,value,count,positive,negative,positive_rate,pred_mean,calibration_gap,auc,pred_std
0,이식된 배아 수_bin,0.0,36544,30,36514,0.000821,0.071417,0.070596,0.550100,0.036001
2,이식된 배아 수_bin,2.0,110845,34483,76362,0.311092,0.586404,0.275312,0.655963,0.219070
3,이식된 배아 수_bin,3.0,8880,1496,7384,0.168468,0.305845,0.137376,0.666380,0.139728
1,이식된 배아 수_bin,1.0,93791,29408,64383,0.313548,0.600063,0.286515,0.684980,0.261081
4,이식된 배아 수_bin,NaN,6291,811,5480,0.128914,0.249524,0.120610,0.685870,0.101038



Numeric Slice: 총 생성 배아 수


,col,value,count,positive,negative,positive_rate,pred_mean,calibration_gap,auc,pred_std
2,총 생성 배아 수_bin,"(5.0, 9.0]",59403,20149,39254,0.339192,0.654290,0.315098,0.677120,0.262573
4,총 생성 배아 수_bin,NaN,6291,811,5480,0.128914,0.249524,0.120610,0.685870,0.101038
1,총 생성 배아 수_bin,"(3.0, 5.0]",42022,12227,29795,0.290967,0.561790,0.270824,0.686267,0.252476
3,총 생성 배아 수_bin,"(9.0, 51.0]",39921,13904,26017,0.348288,0.667119,0.318832,0.711014,0.315802
0,총 생성 배아 수_bin,"(-0.001, 3.0]",108714,19137,89577,0.176031,0.344940,0.168910,0.746377,0.207673



Numeric Slice: 저장된 배아 수


...


Numeric Slice: 배아 해동 경과일_performed


,col,value,count,positive,negative,positive_rate,pred_mean,calibration_gap,auc,pred_std
1,배아 해동 경과일_performed_bin,1,40369,9229,31140,0.228616,0.427800,0.199184,0.656018,0.167664
0,배아 해동 경과일_performed_bin,0,215982,56999,158983,0.263906,0.513497,0.249591,0.750147,0.302943



Numeric Slice: 난자 혼합 경과일_performed


,col,value,count,positive,negative,positive_rate,pred_mean,calibration_gap,auc,pred_std
0,난자 혼합 경과일_performed_bin,0,53735,10155,43580,0.188983,0.364300,0.175317,0.721915,0.194545
1,난자 혼합 경과일_performed_bin,1,202616,56073,146543,0.276745,0.535991,0.259246,0.736322,0.297399


,col,value,count,positive,negative,positive_rate,pred_mean,calibration_gap,auc,pred_std,source_numeric_col
0,이식된 배아 수_bin,0.0,36544,30,36514,0.000821,0.071417,0.070596,0.550100,0.036001,이식된 배아 수
35,배아 이식 경과일_bin,1.0,6053,1132,4921,0.187015,0.370344,0.183330,0.606298,0.119561,배아 이식 경과일
54,배아_이식_집중도_bin,"(0.167, 0.667]",57172,24175,32997,0.422847,0.804224,0.381377,0.608740,0.163511,배아_이식_집중도
36,배아 이식 경과일_bin,0.0,24904,6252,18652,0.251044,0.453350,0.202306,0.615225,0.143071,배아 이식 경과일
37,배아 이식 경과일_bin,5.0,81459,32946,48513,0.404449,0.767590,0.363141,0.619864,0.186849,배아 이식 경과일
38,배아 이식 경과일_bin,6.0,2773,832,1941,0.300036,0.587765,0.287729,0.624893,0.199201,배아 이식 경과일
39,배아 이식 경과일_bin,4.0,4504,1551,2953,0.344361,0.666717,0.322357,0.625448,0.185063,배아 이식 경과일
61,배아_이식률_bin,"(0.111, 0.25]",57534,22385,35149,0.389074,0.738783,0.349709,0.627238,0.188458,배아_이식률
62,배아_이식률_bin,"(0.25, 0.5]",52879,16277,36602,0.307816,0.587119,0.279303,0.644432,0.210972,배아_이식률
43,배아 해동 경과일_bin,0.0,39801,9141,30660,0.229668,0.429720,0.200052,0.653998,0.166080,배아 해동 경과일


In [12]:
all_slice_df = pd.concat(
    [cat_report_df, num_report_df],
    ignore_index=True
)

all_slice_df["abs_calibration_gap"] = all_slice_df["calibration_gap"].abs()

all_slice_df.to_csv(SAVE_DIR / "all_slice_report.csv", index=False)

print("AUC 낮은 slice")
display(
    all_slice_df.sort_values(
        ["auc", "count"],
        ascending=[True, False],
        na_position="last"
    ).head(50)
)

print("Calibration gap 큰 slice")
display(
    all_slice_df.sort_values(
        ["abs_calibration_gap", "count"],
        ascending=[False, False],
        na_position="last"
    ).head(50)
)

AUC 낮은 slice


,col,value,count,positive,negative,positive_rate,pred_mean,calibration_gap,auc,pred_std,source_numeric_col,abs_calibration_gap
45,배아 생성 주요 이유,배아 저장용,9192,8,9184,0.000870,0.042063,0.041192,0.448688,0.026437,NaN,0.041192
63,이식된 배아 수_bin,0.0,36544,30,36514,0.000821,0.071417,0.070596,0.550100,0.036001,이식된 배아 수,0.070596
98,배아 이식 경과일_bin,1.0,6053,1132,4921,0.187015,0.370344,0.183330,0.606298,0.119561,배아 이식 경과일,0.183330
117,배아_이식_집중도_bin,"(0.167, 0.667]",57172,24175,32997,0.422847,0.804224,0.381377,0.608740,0.163511,배아_이식_집중도,0.381377
99,배아 이식 경과일_bin,0.0,24904,6252,18652,0.251044,0.453350,0.202306,0.615225,0.143071,배아 이식 경과일,0.202306
60,대리모 여부,1.0,1049,303,746,0.288847,0.568480,0.279634,0.616803,0.198489,NaN,0.279634
100,배아 이식 경과일_bin,5.0,81459,32946,48513,0.404449,0.767590,0.363141,0.619864,0.186849,배아 이식 경과일,0.363141
101,배아 이식 경과일_bin,6.0,2773,832,1941,0.300036,0.587765,0.287729,0.624893,0.199201,배아 이식 경과일,0.287729
102,배아 이식 경과일_bin,4.0,4504,1551,2953,0.344361,0.666717,0.322357,0.625448,0.185063,배아 이식 경과일,0.322357
124,배아_이식률_bin,"(0.111, 0.25]",57534,22385,35149,0.389074,0.738783,0.349709,0.627238,0.188458,배아_이식률,0.349709


Calibration gap 큰 slice


,col,value,count,positive,negative,positive_rate,pred_mean,calibration_gap,auc,pred_std,source_numeric_col,abs_calibration_gap
117,배아_이식_집중도_bin,"(0.167, 0.667]",57172,24175,32997,0.422847,0.804224,0.381377,0.608740,0.163511,배아_이식_집중도,0.381377
100,배아 이식 경과일_bin,5.0,81459,32946,48513,0.404449,0.767590,0.363141,0.619864,0.186849,배아 이식 경과일,0.363141
124,배아_이식률_bin,"(0.111, 0.25]",57534,22385,35149,0.389074,0.738783,0.349709,0.627238,0.188458,배아_이식률,0.349709
46,배아 생성 주요 이유,"기증용, 현재 시술용",3784,1437,2347,0.379757,0.709398,0.329642,0.671984,0.268099,NaN,0.329642
9,특정 시술 유형,ICSI / BLASTOCYST,1609,574,1035,0.356743,0.686063,0.329320,0.644350,0.235520,NaN,0.329320
102,배아 이식 경과일_bin,4.0,4504,1551,2953,0.344361,0.666717,0.322357,0.625448,0.185063,배아 이식 경과일,0.322357
71,총 생성 배아 수_bin,"(9.0, 51.0]",39921,13904,26017,0.348288,0.667119,0.318832,0.711014,0.315802,총 생성 배아 수,0.318832
68,총 생성 배아 수_bin,"(5.0, 9.0]",59403,20149,39254,0.339192,0.654290,0.315098,0.677120,0.262573,총 생성 배아 수,0.315098
83,혼합된 난자 수_bin,"(9.0, 13.0]",43496,14712,28784,0.338238,0.646546,0.308308,0.683138,0.268419,혼합된 난자 수,0.308308
91,미세주입된 난자 수_bin,"(9.0, 51.0]",43616,14544,29072,0.333456,0.641121,0.307665,0.706044,0.296019,미세주입된 난자 수,0.307665


In [13]:
def add_pair_combo(df, col1, col2):
    combo_name = f"{col1}__x__{col2}"
    df[combo_name] = df[col1].astype(str) + "__" + df[col2].astype(str)
    return combo_name


pair_cols = [
    ("시술 당시 나이", "난자 출처"),
    ("시술 당시 나이", "시술 유형"),
    ("시술 당시 나이", "특정 시술 유형"),
    ("시술 유형", "난자 출처"),
    ("시술 유형", "정자 출처"),
    ("난자 출처", "정자 출처"),
    ("시술 유형", "배아 생성 주요 이유"),
    ("시술 당시 나이", "배아 생성 주요 이유"),
]

pair_reports = []

for col1, col2 in pair_cols:
    if col1 not in analysis_df.columns or col2 not in analysis_df.columns:
        continue

    combo_col = add_pair_combo(analysis_df, col1, col2)

    print("\n" + "=" * 80)
    print("Pair Slice:", combo_col)

    report = slice_auc_report(
        analysis_df,
        col=combo_col,
        min_count=500,
        top_n=50,
    )

    display(report)

    if not report.empty:
        pair_reports.append(report)

pair_report_df = pd.concat(pair_reports, ignore_index=True)
pair_report_df.to_csv(SAVE_DIR / "pair_slice_auc_report.csv", index=False)

display(
    pair_report_df.sort_values(
        ["auc", "count"],
        ascending=[True, False],
        na_position="last"
    ).head(50)
)


Pair Slice: 시술 당시 나이__x__난자 출처


,col,value,count,positive,negative,positive_rate,pred_mean,calibration_gap,auc,pred_std
8,시술 당시 나이__x__난자 출처,만38-39세__알 수 없음,1053,108,945,0.102564,0.209079,0.106515,0.540849,0.038561
2,시술 당시 나이__x__난자 출처,만18-34세__알 수 없음,2071,403,1668,0.194592,0.326186,0.131594,0.591422,0.108857
5,시술 당시 나이__x__난자 출처,만35-37세__알 수 없음,1450,219,1231,0.151034,0.272573,0.121538,0.613901,0.081315
11,시술 당시 나이__x__난자 출처,만40-42세__알 수 없음,1036,72,964,0.069498,0.174440,0.104942,0.627622,0.020815
0,시술 당시 나이__x__난자 출처,만18-34세__기증 제공,2871,880,1991,0.306513,0.617108,0.310594,0.665797,0.238441
12,시술 당시 나이__x__난자 출처,만43-44세__기증 제공,2459,837,1622,0.340382,0.631100,0.290718,0.667978,0.257793
3,시술 당시 나이__x__난자 출처,만35-37세__기증 제공,1869,594,1275,0.317817,0.603854,0.286037,0.677184,0.250966
14,시술 당시 나이__x__난자 출처,만45-50세__기증 제공,3679,1075,2604,0.292199,0.565240,0.273041,0.685464,0.249755
6,시술 당시 나이__x__난자 출처,만38-39세__기증 제공,1619,525,1094,0.324274,0.604799,0.280525,0.686917,0.245488
9,시술 당시 나이__x__난자 출처,만40-42세__기증 제공,3272,1063,2209,0.324878,0.615519,0.290642,0.699801,0.252916



Pair Slice: 시술 당시 나이__x__시술 유형


,col,value,count,positive,negative,positive_rate,pred_mean,calibration_gap,auc,pred_std
4,시술 당시 나이__x__시술 유형,만38-39세__DI,1053,108,945,0.102564,0.209079,0.106515,0.540849,0.038561
0,시술 당시 나이__x__시술 유형,만18-34세__DI,2071,403,1668,0.194592,0.326186,0.131594,0.591422,0.108857
2,시술 당시 나이__x__시술 유형,만35-37세__DI,1450,219,1231,0.151034,0.272573,0.121538,0.613901,0.081315
6,시술 당시 나이__x__시술 유형,만40-42세__DI,1036,72,964,0.069498,0.174440,0.104942,0.627622,0.020815
3,시술 당시 나이__x__시술 유형,만35-37세__IVF,56330,15867,40463,0.281679,0.546205,0.264526,0.700213,0.260425
1,시술 당시 나이__x__시술 유형,만18-34세__IVF,100405,32658,67747,0.325263,0.627744,0.302481,0.702743,0.284783
5,시술 당시 나이__x__시술 유형,만38-39세__IVF,38194,8414,29780,0.220296,0.420648,0.200351,0.707669,0.225349
7,시술 당시 나이__x__시술 유형,만40-42세__IVF,36312,5881,30431,0.161957,0.309306,0.147348,0.735787,0.197537
9,시술 당시 나이__x__시술 유형,만45-50세__IVF,6656,1159,5497,0.174129,0.365818,0.191690,0.825803,0.293151
8,시술 당시 나이__x__시술 유형,만43-44세__IVF,11834,1438,10396,0.121514,0.259644,0.138130,0.826452,0.235245



Pair Slice: 시술 당시 나이__x__특정 시술 유형


...


Pair Slice: 시술 유형__x__배아 생성 주요 이유


,col,value,count,positive,negative,positive_rate,pred_mean,calibration_gap,auc,pred_std
3,시술 유형__x__배아 생성 주요 이유,IVF__배아 저장용,9192,8,9184,0.000870,0.042063,0.041192,0.448688,0.026437
1,시술 유형__x__배아 생성 주요 이유,"IVF__기증용, 현재 시술용",3784,1437,2347,0.379757,0.709398,0.329642,0.671984,0.268099
5,시술 유형__x__배아 생성 주요 이유,NaN,6291,811,5480,0.128914,0.249524,0.120610,0.685870,0.101038
4,시술 유형__x__배아 생성 주요 이유,IVF__현재 시술용,233732,63942,169790,0.273570,0.527529,0.253959,0.719926,0.273973
2,시술 유형__x__배아 생성 주요 이유,IVF__난자 저장용,1959,0,1959,0.000000,0.056388,0.056388,NaN,0.023990
0,시술 유형__x__배아 생성 주요 이유,IVF__기증용,1108,0,1108,0.000000,0.054112,0.054112,NaN,0.032923



Pair Slice: 시술 당시 나이__x__배아 생성 주요 이유


,col,value,count,positive,negative,positive_rate,pred_mean,calibration_gap,auc,pred_std
3,시술 당시 나이__x__배아 생성 주요 이유,만18-34세__배아 저장용,3169,4,3165,0.001262,0.048917,0.047654,0.100632,0.019672
8,시술 당시 나이__x__배아 생성 주요 이유,만38-39세__배아 저장용,1249,1,1248,0.000801,0.043962,0.043161,0.491186,0.024774
1,시술 당시 나이__x__배아 생성 주요 이유,"만18-34세__기증용, 현재 시술용",3280,1279,2001,0.389939,0.725741,0.335802,0.660888,0.264857
7,시술 당시 나이__x__배아 생성 주요 이유,만35-37세__현재 시술용,53233,15705,37528,0.295024,0.569439,0.274415,0.679152,0.242461
4,시술 당시 나이__x__배아 생성 주요 이유,만18-34세__현재 시술용,92805,31358,61447,0.337891,0.650697,0.312806,0.682889,0.262984
0,시술 당시 나이__x__배아 생성 주요 이유,NaN,6291,811,5480,0.128914,0.249524,0.120610,0.685870,0.101038
9,시술 당시 나이__x__배아 생성 주요 이유,만38-39세__현재 시술용,36416,8410,28006,0.230942,0.438658,0.207716,0.689473,0.214859
11,시술 당시 나이__x__배아 생성 주요 이유,만40-42세__현재 시술용,34113,5875,28238,0.172222,0.327128,0.154906,0.715532,0.190225
12,시술 당시 나이__x__배아 생성 주요 이유,만43-44세__배아 저장용,743,2,741,0.002692,0.031836,0.029145,0.738192,0.032456
14,시술 당시 나이__x__배아 생성 주요 이유,만45-50세__현재 시술용,6202,1159,5043,0.186875,0.390336,0.203460,0.810150,0.288632


,col,value,count,positive,negative,positive_rate,pred_mean,calibration_gap,auc,pred_std
69,시술 당시 나이__x__배아 생성 주요 이유,만18-34세__배아 저장용,3169,4,3165,0.001262,0.048917,0.047654,0.100632,0.019672
63,시술 유형__x__배아 생성 주요 이유,IVF__배아 저장용,9192,8,9184,0.000870,0.042063,0.041192,0.448688,0.026437
70,시술 당시 나이__x__배아 생성 주요 이유,만38-39세__배아 저장용,1249,1,1248,0.000801,0.043962,0.043161,0.491186,0.024774
0,시술 당시 나이__x__난자 출처,만38-39세__알 수 없음,1053,108,945,0.102564,0.209079,0.106515,0.540849,0.038561
16,시술 당시 나이__x__시술 유형,만38-39세__DI,1053,108,945,0.102564,0.209079,0.106515,0.540849,0.038561
26,시술 당시 나이__x__특정 시술 유형,만38-39세__IUI,1031,105,926,0.101843,0.209585,0.107742,0.548853,0.038710
27,시술 당시 나이__x__특정 시술 유형,만18-34세__IUI,1976,383,1593,0.193826,0.327971,0.134145,0.587864,0.108959
1,시술 당시 나이__x__난자 출처,만18-34세__알 수 없음,2071,403,1668,0.194592,0.326186,0.131594,0.591422,0.108857
17,시술 당시 나이__x__시술 유형,만18-34세__DI,2071,403,1668,0.194592,0.326186,0.131594,0.591422,0.108857
28,시술 당시 나이__x__특정 시술 유형,만35-37세__IUI,1411,215,1196,0.152374,0.273238,0.120864,0.609835,0.081431


In [14]:
# 실제 1인데 낮게 예측한 케이스
false_negative_like = analysis_df[
    analysis_df["target"] == 1
].sort_values("pred", ascending=True)

# 실제 0인데 높게 예측한 케이스
false_positive_like = analysis_df[
    analysis_df["target"] == 0
].sort_values("pred", ascending=False)

cols_to_view = [
    "target",
    "pred",
    "pred_rank",
    "시술 유형",
    "특정 시술 유형",
    "시술 당시 나이",
    "난자 출처",
    "정자 출처",
    "배아 생성 주요 이유",
    "이식된 배아 수",
    "총 생성 배아 수",
    "저장된 배아 수",
    "수집된 신선 난자 수",
    "배아 이식 경과일",
]

cols_to_view = [col for col in cols_to_view if col in analysis_df.columns]

print("실제 성공인데 낮게 예측한 샘플")
display(false_negative_like[cols_to_view].head(30))

print("실제 실패인데 높게 예측한 샘플")
display(false_positive_like[cols_to_view].head(30))

false_negative_like[cols_to_view].head(200).to_csv(
    SAVE_DIR / "false_negative_like_top200.csv",
    index=False
)

false_positive_like[cols_to_view].head(200).to_csv(
    SAVE_DIR / "false_positive_like_top200.csv",
    index=False
)

실제 성공인데 낮게 예측한 샘플


,target,pred,pred_rank,시술 유형,특정 시술 유형,시술 당시 나이,난자 출처,정자 출처,배아 생성 주요 이유,이식된 배아 수,총 생성 배아 수,저장된 배아 수,수집된 신선 난자 수,배아 이식 경과일
102235,1,0.012556,0.006807,IVF,ICSI,만43-44세,본인 제공,배우자 제공,현재 시술용,0.0,0.0,0.0,4.0,NaN
49578,1,0.016360,0.009487,IVF,ICSI:ICSI,만18-34세,본인 제공,배우자 제공,배아 저장용,0.0,15.0,9.0,18.0,NaN
212604,1,0.017479,0.010294,IVF,ICSI,만43-44세,본인 제공,기증 제공,배아 저장용,0.0,7.0,2.0,12.0,NaN
78940,1,0.024081,0.015666,IVF,ICSI,만38-39세,본인 제공,배우자 제공,현재 시술용,0.0,6.0,6.0,16.0,NaN
107763,1,0.024969,0.016450,IVF,ICSI,만18-34세,본인 제공,배우자 제공,배아 저장용,0.0,6.0,5.0,8.0,NaN
116444,1,0.028499,0.019793,IVF,ICSI,만18-34세,본인 제공,배우자 제공,배아 저장용,0.0,14.0,11.0,23.0,NaN
227626,1,0.033403,0.024747,IVF,ICSI,만18-34세,본인 제공,배우자 제공,배아 저장용,0.0,9.0,9.0,19.0,NaN
55659,1,0.035111,0.026741,IVF,IVF:IVF,만38-39세,본인 제공,배우자 제공,배아 저장용,0.0,11.0,2.0,20.0,NaN
136155,1,0.045120,0.038724,IVF,ICSI,만38-39세,본인 제공,배우자 제공,현재 시술용,0.0,5.0,3.0,11.0,NaN
57176,1,0.052889,0.048188,IVF,ICSI,만35-37세,본인 제공,배우자 제공,현재 시술용,0.0,5.0,1.0,8.0,NaN


실제 실패인데 높게 예측한 샘플


,target,pred,pred_rank,시술 유형,특정 시술 유형,시술 당시 나이,난자 출처,정자 출처,배아 생성 주요 이유,이식된 배아 수,총 생성 배아 수,저장된 배아 수,수집된 신선 난자 수,배아 이식 경과일
117522,0,0.999936,0.999977,IVF,ICSI,만18-34세,본인 제공,배우자 제공,현재 시술용,2.0,10.0,6.0,17.0,5.0
61327,0,0.999934,0.999973,IVF,ICSI / BLASTOCYST,만18-34세,본인 제공,배우자 제공,현재 시술용,2.0,11.0,5.0,11.0,5.0
242223,0,0.999894,0.999965,IVF,ICSI,만35-37세,본인 제공,배우자 제공,현재 시술용,2.0,15.0,4.0,17.0,5.0
149996,0,0.999894,0.999961,IVF,IVF,만18-34세,본인 제공,배우자 제공,현재 시술용,2.0,18.0,5.0,19.0,5.0
181485,0,0.999807,0.999926,IVF,ICSI,만18-34세,본인 제공,배우자 제공,"기증용, 현재 시술용",2.0,7.0,4.0,19.0,5.0
146842,0,0.999806,0.999922,IVF,ICSI,만18-34세,본인 제공,배우자 제공,현재 시술용,2.0,14.0,3.0,17.0,5.0
104443,0,0.999798,0.999914,IVF,IVF,만18-34세,본인 제공,기증 제공,현재 시술용,2.0,8.0,2.0,10.0,5.0
145517,0,0.999759,0.999883,IVF,ICSI,만18-34세,본인 제공,배우자 제공,"기증용, 현재 시술용",2.0,7.0,3.0,15.0,5.0
183247,0,0.999735,0.999856,IVF,ICSI,만18-34세,본인 제공,배우자 제공,현재 시술용,2.0,20.0,12.0,24.0,5.0
196789,0,0.999731,0.999848,IVF,ICSI,만18-34세,본인 제공,배우자 제공,현재 시술용,2.0,15.0,6.0,17.0,5.0


In [15]:
candidate_slices = all_slice_df.copy()

candidate_slices = candidate_slices[
    (candidate_slices["count"] >= 1000) &
    (candidate_slices["auc"].notna())
].copy()

candidate_slices["weakness_score"] = (
    (0.7407705075 - candidate_slices["auc"]) *
    np.log1p(candidate_slices["count"])
)

candidate_slices = candidate_slices.sort_values(
    "weakness_score",
    ascending=False
)

display(candidate_slices.head(50))

candidate_slices.to_csv(
    SAVE_DIR / "candidate_weak_slices.csv",
    index=False
)

,col,value,count,positive,negative,positive_rate,pred_mean,calibration_gap,auc,pred_std,source_numeric_col,abs_calibration_gap,weakness_score
45,배아 생성 주요 이유,배아 저장용,9192,8,9184,0.000870,0.042063,0.041192,0.448688,0.026437,NaN,0.041192,2.665603
63,이식된 배아 수_bin,0.0,36544,30,36514,0.000821,0.071417,0.070596,0.550100,0.036001,이식된 배아 수,0.070596,2.003237
117,배아_이식_집중도_bin,"(0.167, 0.667]",57172,24175,32997,0.422847,0.804224,0.381377,0.608740,0.163511,배아_이식_집중도,0.381377,1.446244
100,배아 이식 경과일_bin,5.0,81459,32946,48513,0.404449,0.767590,0.363141,0.619864,0.186849,배아 이식 경과일,0.363141,1.367197
99,배아 이식 경과일_bin,0.0,24904,6252,18652,0.251044,0.453350,0.202306,0.615225,0.143071,배아 이식 경과일,0.202306,1.270877
124,배아_이식률_bin,"(0.111, 0.25]",57534,22385,35149,0.389074,0.738783,0.349709,0.627238,0.188458,배아_이식률,0.349709,1.244329
98,배아 이식 경과일_bin,1.0,6053,1132,4921,0.187015,0.370344,0.183330,0.606298,0.119561,배아 이식 경과일,0.183330,1.171051
125,배아_이식률_bin,"(0.25, 0.5]",52879,16277,36602,0.307816,0.587119,0.279303,0.644432,0.210972,배아_이식률,0.279303,1.047760
118,배아_이식_집중도_bin,"(0.667, 1.0]",147652,37163,110489,0.251693,0.477515,0.225822,0.657938,0.198239,배아_이식_집중도,0.225822,0.985919
64,이식된 배아 수_bin,2.0,110845,34483,76362,0.311092,0.586404,0.275312,0.655963,0.219070,이식된 배아 수,0.275312,0.985113


In [16]:
import pandas as pd
from pathlib import Path

SAVE_DIR = Path("/mnt/c/dev/my_ml_project/analysis/oof_slice")

all_slice_df = pd.read_csv(SAVE_DIR / "all_slice_report.csv")
candidate_slices = pd.read_csv(SAVE_DIR / "candidate_weak_slices.csv")
pair_slice_df = pd.read_csv(SAVE_DIR / "pair_slice_auc_report.csv")

print(all_slice_df.shape)
print(candidate_slices.shape)
print(pair_slice_df.shape)

(146, 12)
(142, 13)
(84, 10)


In [17]:
# 샘플 수 충분 + AUC 낮은 slice
weak_slices = all_slice_df[
    (all_slice_df["count"] >= 3000) &
    (all_slice_df["auc"].notna()) &
    (all_slice_df["auc"] < 0.72)
].copy()

weak_slices = weak_slices.sort_values(
    ["auc", "count"],
    ascending=[True, False]
)

display(weak_slices.head(50))

,col,value,count,positive,negative,positive_rate,pred_mean,calibration_gap,auc,pred_std,source_numeric_col,abs_calibration_gap
45,배아 생성 주요 이유,배아 저장용,9192,8,9184,0.000870,0.042063,0.041192,0.448688,0.026437,NaN,0.041192
63,이식된 배아 수_bin,0.0,36544,30,36514,0.000821,0.071417,0.070596,0.550100,0.036001,이식된 배아 수,0.070596
98,배아 이식 경과일_bin,1.0,6053,1132,4921,0.187015,0.370344,0.183330,0.606298,0.119561,배아 이식 경과일,0.183330
117,배아_이식_집중도_bin,"(0.167, 0.667]",57172,24175,32997,0.422847,0.804224,0.381377,0.608740,0.163511,배아_이식_집중도,0.381377
99,배아 이식 경과일_bin,0.0,24904,6252,18652,0.251044,0.453350,0.202306,0.615225,0.143071,배아 이식 경과일,0.202306
100,배아 이식 경과일_bin,5.0,81459,32946,48513,0.404449,0.767590,0.363141,0.619864,0.186849,배아 이식 경과일,0.363141
102,배아 이식 경과일_bin,4.0,4504,1551,2953,0.344361,0.666717,0.322357,0.625448,0.185063,배아 이식 경과일,0.322357
124,배아_이식률_bin,"(0.111, 0.25]",57534,22385,35149,0.389074,0.738783,0.349709,0.627238,0.188458,배아_이식률,0.349709
125,배아_이식률_bin,"(0.25, 0.5]",52879,16277,36602,0.307816,0.587119,0.279303,0.644432,0.210972,배아_이식률,0.279303
51,신선 배아 사용 여부,0.0,39924,9166,30758,0.229586,0.429630,0.200043,0.653818,0.166171,NaN,0.200043


In [18]:
weak_pair_slices = pair_slice_df[
    (pair_slice_df["count"] >= 1000) &
    (pair_slice_df["auc"].notna()) &
    (pair_slice_df["auc"] < 0.72)
].copy()

weak_pair_slices = weak_pair_slices.sort_values(
    ["auc", "count"],
    ascending=[True, False]
)

display(weak_pair_slices.head(50))

,col,value,count,positive,negative,positive_rate,pred_mean,calibration_gap,auc,pred_std
69,시술 당시 나이__x__배아 생성 주요 이유,만18-34세__배아 저장용,3169,4,3165,0.001262,0.048917,0.047654,0.100632,0.019672
63,시술 유형__x__배아 생성 주요 이유,IVF__배아 저장용,9192,8,9184,0.000870,0.042063,0.041192,0.448688,0.026437
70,시술 당시 나이__x__배아 생성 주요 이유,만38-39세__배아 저장용,1249,1,1248,0.000801,0.043962,0.043161,0.491186,0.024774
0,시술 당시 나이__x__난자 출처,만38-39세__알 수 없음,1053,108,945,0.102564,0.209079,0.106515,0.540849,0.038561
16,시술 당시 나이__x__시술 유형,만38-39세__DI,1053,108,945,0.102564,0.209079,0.106515,0.540849,0.038561
26,시술 당시 나이__x__특정 시술 유형,만38-39세__IUI,1031,105,926,0.101843,0.209585,0.107742,0.548853,0.038710
27,시술 당시 나이__x__특정 시술 유형,만18-34세__IUI,1976,383,1593,0.193826,0.327971,0.134145,0.587864,0.108959
1,시술 당시 나이__x__난자 출처,만18-34세__알 수 없음,2071,403,1668,0.194592,0.326186,0.131594,0.591422,0.108857
17,시술 당시 나이__x__시술 유형,만18-34세__DI,2071,403,1668,0.194592,0.326186,0.131594,0.591422,0.108857
28,시술 당시 나이__x__특정 시술 유형,만35-37세__IUI,1411,215,1196,0.152374,0.273238,0.120864,0.609835,0.081431


In [19]:
gap_slices = all_slice_df.copy()
gap_slices["abs_calibration_gap"] = gap_slices["calibration_gap"].abs()

gap_slices = gap_slices[
    gap_slices["count"] >= 3000
].sort_values(
    "abs_calibration_gap",
    ascending=False
)

display(gap_slices.head(50))

,col,value,count,positive,negative,positive_rate,pred_mean,calibration_gap,auc,pred_std,source_numeric_col,abs_calibration_gap
117,배아_이식_집중도_bin,"(0.167, 0.667]",57172,24175,32997,0.422847,0.804224,0.381377,0.608740,0.163511,배아_이식_집중도,0.381377
100,배아 이식 경과일_bin,5.0,81459,32946,48513,0.404449,0.767590,0.363141,0.619864,0.186849,배아 이식 경과일,0.363141
124,배아_이식률_bin,"(0.111, 0.25]",57534,22385,35149,0.389074,0.738783,0.349709,0.627238,0.188458,배아_이식률,0.349709
46,배아 생성 주요 이유,"기증용, 현재 시술용",3784,1437,2347,0.379757,0.709398,0.329642,0.671984,0.268099,NaN,0.329642
102,배아 이식 경과일_bin,4.0,4504,1551,2953,0.344361,0.666717,0.322357,0.625448,0.185063,배아 이식 경과일,0.322357
71,총 생성 배아 수_bin,"(9.0, 51.0]",39921,13904,26017,0.348288,0.667119,0.318832,0.711014,0.315802,총 생성 배아 수,0.318832
68,총 생성 배아 수_bin,"(5.0, 9.0]",59403,20149,39254,0.339192,0.654290,0.315098,0.677120,0.262573,총 생성 배아 수,0.315098
83,혼합된 난자 수_bin,"(9.0, 13.0]",43496,14712,28784,0.338238,0.646546,0.308308,0.683138,0.268419,혼합된 난자 수,0.308308
91,미세주입된 난자 수_bin,"(9.0, 51.0]",43616,14544,29072,0.333456,0.641121,0.307665,0.706044,0.296019,미세주입된 난자 수,0.307665
96,파트너 정자와 혼합된 난자 수_bin,"(12.0, 51.0]",47556,15679,31877,0.329696,0.635787,0.306091,0.713443,0.307795,파트너 정자와 혼합된 난자 수,0.306091
